# Cinemática Inversa Numérica
---

In [90]:
import numpy as np
import itertools
from numpy import float32, float64, int8, int32, int64

from dataclasses import dataclass
from typing import Self

Utilizamos alguna semilla fila, para que las pruebas a pesar de ser aleatoria, sean determinista para cada corrida del código y utilizar los resultados estadísticos

In [91]:
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

In [92]:
CANT_EJES = 6

## Definición del Robot
---
En esta sección vamos a definir la Pose, el Robot IRB140 con su configuración y dimensiones, para usarlos a lo largo del trabajo. Se define la cinemática directa del mismo ya que será necesario para calcula la cinemática inversa de forma numérica

In [93]:
def problema_inverso_rotacion(rotacion: np.ndarray, epsilon: float64 = 1e-9) -> np.ndarray:
  theta = np.arccos((np.trace(rotacion) - 1) / 2)

  sen_theta = np.sin(theta)
  if abs(sen_theta) > epsilon:
    kx = (rotacion[2, 1] - rotacion[1, 2]) / (2 * sen_theta)
    ky = (rotacion[0, 2] - rotacion[2, 0]) / (2 * sen_theta)
    kz = (rotacion[1, 0] - rotacion[0, 1]) / (2 * sen_theta)

  else:
    kx = np.sqrt((rotacion[0, 0] + 1) / 2)
    ky = np.sqrt((rotacion[1, 1] + 1) / 2)
    kz = np.sqrt((rotacion[2, 2] + 1) / 2)

  k = np.array([kx, ky, kz])
  return theta * k / np.linalg.norm(k)

def matriz_antisimetrica(vector: np.ndarray) -> np.ndarray:
  x, y, z = tuple(vector.tolist())
  return np.array([
      [  0, -z,  y ],
      [  z,  0, -x ],
      [ -y,  x,  0 ],
  ])

In [94]:
@dataclass
class Pose:
  rotacion: np.ndarray
  translacion: np.ndarray

  @classmethod
  def identidad(cls) -> Self:
    return Pose(np.eye(3), np.zeros(3))

  def __mul__(self, pose: Self) -> Self:
    return Pose(
      self.rotacion @ pose.rotacion,
      self.rotacion @ pose.translacion + self.translacion,
    )

  def __sub__(self, pose: Self) -> np.ndarray:
    diff_rotacion = problema_inverso_rotacion(pose.rotacion @ self.rotacion.T)
    diff_translacion = self.translacion - pose.translacion
    return np.hstack(( diff_translacion, diff_rotacion ))

  def inv(self) -> Self:
    return Pose(
        self.rotacion.T,
        -self.rotacion.T @ self.translacion,
    )

In [95]:
def DH(theta: float64, d: float64, a: float64, alpha: float64) -> Pose:
    cth = np.cos(theta)
    sth = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    return Pose(np.array([
        [cth, -sth*ca,  sth*sa],
        [sth,  cth*ca, -cth*sa],
        [  0,     sa,       ca],
    ]), np.array([ a*cth, a*sth, d ]))

In [96]:
@dataclass
class Dimensiones:
  # Representan las dimensiones en metros
  d1: float64
  a1: float64
  a2: float64
  d4: float64
  d6: float64

@dataclass
class Configuracion:
  # True es lo que vimos como 1, y Flase como -1
  c1: bool
  c2: bool
  c3: bool

  def __eq__(self, otro: Self) -> bool:
    return self.c1 == otro.c1 and self.c2 == otro.c2 and self.c3 == otro.c3

class IRB140:
  def __init__(self, dim: Dimensiones, ABase0: Pose | None = None, A6Tool: Pose | None = None):
    self.dim = dim

    self.A01 = lambda q1: DH(q1, dim.d1, dim.a1, -np.pi/2)
    self.A12 = lambda q2: DH(q2,      0, dim.a2,        0)
    self.A23 = lambda q3: DH(q3,      0,      0,  np.pi/2)
    self.A34 = lambda q4: DH(q4, dim.d4,      0, -np.pi/2)
    self.A45 = lambda q5: DH(q5,      0,      0,  np.pi/2)
    self.A56 = lambda q6: DH(q6, dim.d6,      0,        0)

    self.ABase0 = Pose.identidad() if ABase0 is None else ABase0
    self.A6Tool = Pose.identidad() if A6Tool is None else A6Tool

  def cinematica_directa(self, q: np.ndarray) -> Pose:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())
    A06 = self.A01(q1) * self.A12(q2) * self.A23(q3) * self.A34(q4) * self.A45(q5) * self.A56(q6)
    return self.ABase0 * A06 * self.A6Tool

  def cinematica_velocidad(self, q: np.ndarray) -> np.ndarray:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())

    As = [self.A01(q1), self.A12(q2), self.A23(q3), self.A34(q4), self.A45(q5), self.A56(q6)]
    Aacc = Pose.identidad()

    Jn = np.zeros((CANT_EJES, CANT_EJES))
    for i in range(CANT_EJES - 1, -1, -1):
      # Aacc = I
      # necesito generar A^n_{n-1}, A^n_{n-2}, ..., A^n_0
      # Aacc = A^n_{n-2} = A^{n-1}_{n-2} @ A^n_{n-1}
      Aacc = As[i] * Aacc
      p = Aacc.translacion
      x = Aacc.rotacion[:, 0]
      y = Aacc.rotacion[:, 1]
      z = Aacc.rotacion[:, 2]

      for j, versor in enumerate((x, y, z)):
        Jn[j, i] = np.cross(p, versor)[2]
        Jn[j + 3, i] = versor[2]

    RBase6 = (self.ABase0 * Aacc).rotacion
    p6Tool = self.A6Tool.translacion
    JnBase = np.vstack((
        np.hstack(( RBase6, -RBase6 @ matriz_antisimetrica(p6Tool) )),
        np.hstack(( np.zeros(RBase6.shape), RBase6 ))
    ))

    return JnBase @ Jn

  def configuracion(self, q: np.ndarray, epsilon: float64 = 1e-9) -> Configuracion | None:
    _, q2, q3, _, q5, _ = tuple(q.tolist())

    config1 = self.dim.d4 * np.sin(q2 + q3) + self.dim.a2 * np.cos(q2) + self.dim.a1
    config2 = np.cos(q3)
    config3 = q5

    if abs(config1) < epsilon or abs(config2) < epsilon or abs(config3) < epsilon:
      return None

    return Configuracion(bool(config1 > 0), bool(config2 > 0), bool(config3 > 0))

irb140 = IRB140(Dimensiones(0.352, 0.070, 0.360, 0.380, 0))

## Cinemática inversa naïve
---
La forma que plantearemos para resolver la cinemática inversa es a partir de generar una función $f(\bar{q})$, la cual tenga como raiz el valor de $\bar{q}$ que lleve el end-effector a la pose destino $\text{Pose}_{dest}$

Utilizando el método de Newton-Raphson, nos permite encontrar los ceros de una función de forma iterativa de la siguiente forma $$ \bar{q}_{n + 1} = \bar{q}_{n} - \frac{f(\bar{q}_{n})}{f'(\bar{q}_{n})} $$

Por lo tanto, para resolver la busqueda de raices, plantearemos la función $$
  f(\bar{q}) = \text{Pose}_{dest} - \text{Pose}_{base}^{tool}(\bar{q})
$$ donde la derivada la podemos aproximar como $$
  f'(\bar{q}) = -J_{base}^{tool}(\bar{q})
$$

Esto en conjunto, dando la expresión $$
  \bar{q}_{n + 1} = \bar{q}_{n} + \alpha \left( J_{base}^{tool}(\bar{q}) \right)^\dagger \cdot \left( \text{Pose}_{dest} - \text{Pose}_{base}^{tool}(\bar{q}) \right)
$$ donde $\alpha$ sería un hiperparámetro que permite ajustar la velocidad que avanza cada iteración y $A^\dagger$ es el pseudo-inversa de la matriz $A$

In [97]:
def cinematica_inversa_naive(
  pose_destino: Pose,
  alfa: float64,
  epsilon: float64 = 1e-3,
  max_iteraciones: int32 = 1000,
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo
  q = np.array([0., 0., 0., 0., np.pi / 2, 0.])

  termina_correctamente = False
  for _ in range(max_iteraciones):
    delta_pose = pose_destino - irb140.cinematica_directa(q)
    if np.linalg.norm(delta_pose)**2 < epsilon:
      termina_correctamente = True
      break

    J = irb140.cinematica_velocidad(q)
    Jinv = np.linalg.pinv(J)

    q += alfa * Jinv @ (delta_pose / np.linalg.norm(delta_pose))
    q = np.mod(q + np.pi, 2 * np.pi) - np.pi

  return q, termina_correctamente

Para confirmar el funcionamiento, realizaremos pruebas generando $100$ valores de $\bar{q}$ aleatorios, los cuales nos aseguramos que tenga su correspondiente inversa asegurandonos que tenga una configuración valida, y por cinemática directa calcular su correspondiente pose, la cual será la pose destino utilizada en el calculo de la cinemática inversa

In [98]:
def generar_angulos() -> tuple[np.ndarray, Configuracion]:
  q = np.zeros(CANT_EJES)
  config = None

  while config is None:
    # definimos los q ∈ [-π, π]
    q = 2 * np.pi * (np.random.rand(CANT_EJES) - 0.5)
    config = irb140.configuracion(q)

  return q, config


In [99]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, _ = generar_angulos()
  q_obtenido, _ = cinematica_inversa_naive(irb140.cinematica_directa(q_destino), 0.1, max_iteraciones = 100)

  if np.linalg.norm(q_destino - q_obtenido) < 1e-1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas: 100
	Resultado: 0/100


Se obtiene el resultado de $10$ entre las $100$ pruebas que se intentaron, esto muestra que a pesar de obtener resultados correctos, el algoritmo necesita más información para obtener el $100\%$ de aciertos

Esto ocurre ya que el problema que resuelve el gradiente descendiente es de minimización. Como conocemos el robot, sabemos que para una dada pose pueden existir multiples configuraciones del robot que lleven el end-effector a la pose destino, y esta implementación de la cinemática inversa no puede distinguir entre estas configuraciones dandonos como resultado los $\bar{q}$ más cercanos a la configuración de la semilla inicial

## Cinemática inversa con más información
---
Como vimos se tiene que incorporar más información al algoritmo para obtener el $100\%$ de aciertos. Esta información puede ser dada por la configuración de la pose destino, por lo que probarémos utilizar una semilla que tenga la misma configuración

In [100]:
def semilla_configuracion(config: Configuracion) -> np.ndarray:
  q2 = 0
  if not config.c1:
    alpha = np.atan2(irb140.dim.d4, irb140.dim.a2)
    q2 = alpha - np.pi if config.c2 else np.pi - alpha

  q3 = 0 if config.c2 else np.pi
  q5 = np.pi / 2 if config.c3 else -np.pi / 2

  if irb140.dim.d4 * np.sin(q2 + q3) + irb140.dim.a2 * np.cos(q2) + irb140.dim.a1 > 0 and not config.c1:
    raise ValueError("no se puede obtener esta configuración con las dimensiones del robot")

  return np.array([ 0., q2, q3, 0., q5, 0. ])

In [101]:
def cinematica_inversa_incompleta(
  pose_destino: Pose,
  config: Configuracion,
  alfa: float64,
  epsilon: float64 = 1e-2,
  max_iteraciones: int32 = 1000,
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo determinada por la configuración
  q = semilla_configuracion(config)

  termina_correctamente = False
  for k in range(max_iteraciones):
    delta_pose = pose_destino - irb140.cinematica_directa(q)
    if np.linalg.norm(delta_pose)**2 < epsilon:
      termina_correctamente = True
      break

    J = irb140.cinematica_velocidad(q)
    Jinv = np.linalg.pinv(J)

    q += alfa * Jinv @ (delta_pose / np.linalg.norm(delta_pose))
    q = np.mod(q + np.pi, 2 * np.pi) - np.pi

  return q, termina_correctamente

In [102]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas con configuración: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, config = generar_angulos()
  q_obtenido, _ = cinematica_inversa_incompleta(irb140.cinematica_directa(q_destino), config, 0.5, max_iteraciones = 100)

  if np.linalg.norm(q_destino - q_obtenido) < 1e-1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas con configuración: 100
	Resultado: 0/100


## Cinemática inversa con más información aún
---


In [103]:
def semilla_cuadrante(config: Configuracion, c1: int8, c4: int8, c6: int8) -> np.ndarray:
  _, q2, q3, _, q5, _ = tuple(semilla_configuracion(config).tolist())
  q1, q4, q6 = (
      np.pi / 4 + (cuadrante - 2) * np.pi / 2
      for cuadrante in [c1, c4, c6]
  )
  return np.array([ q1, q2, q3, q4, q5, q6 ])

In [104]:
def cinematica_inversa(
  pose_destino: Pose,
  config: Configuracion,
  alfa: float64,
  epsilon: float64 = 1e-2,
  max_iteraciones: int32 = 1000 / (4 * 4 * 4),
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo determinada por la configuración, y su correspondiente cuadrante

  for c1, c4, c6 in itertools.product(range(4), repeat = 3):
    q = semilla_cuadrante(config, c1, c4, c6)

    termina_correctamente = False
    cambio_cuadrante = False
    for k in range(max_iteraciones):
      delta_pose = pose_destino - irb140.cinematica_directa(q)
      if np.linalg.norm(delta_pose)**2 < epsilon:
        termina_correctamente = True
        break

      J = irb140.cinematica_velocidad(q)
      Jinv = np.linalg.pinv(J)

      q += alfa * Jinv @ (delta_pose / np.linalg.norm(delta_pose))
      q = np.mod(q + np.pi, 2 * np.pi) - np.pi

      nueva_config = irb140.configuracion(q)
      if nueva_config is None or nueva_config != config:
        # print("Cambio de configuración, reintentando en otros cuadrantes
        cambio_cuadrante = True
        break

    if not cambio_cuadrante:
      break

  return q, termina_correctamente

In [105]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas con configuración: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, config = generar_angulos()
  q_obtenido, _ = cinematica_inversa(irb140.cinematica_directa(q_destino), config, 0.5, max_iteraciones = 100)

  if np.linalg.norm(q_destino[:3] - q_obtenido[:3]) < 1e-1:
    cant_acertada += 1
  else:
    print("-" * 100)
    print(f"Esperado: {q_destino} y {config}\nObtenido: {q_obtenido} y {irb140.configuracion(q_obtenido)}")
    print(f"Pose esperada: {irb140.cinematica_directa(q_destino)}\nPose obtenida: {irb140.cinematica_directa(q_obtenido)}")

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas con configuración: 100
----------------------------------------------------------------------------------------------------
Esperado: [ 0.307  1.981 -2.522  1.892 -2.883  1.988] y Configuracion(c1=False, c2=False, c3=False)
Obtenido: [ 0.542  1.829 -2.256 -2.077  0.168 -2.681] y Configuracion(c1=False, c2=False, c3=True)
Pose esperada: Pose(rotacion=array([[-0.784, -0.091,  0.614],
       [-0.161,  0.985, -0.06 ],
       [-0.6  , -0.146, -0.787]]), translacion=array([-0.257, -0.081,  0.347]))
Pose obtenida: Pose(rotacion=array([[-0.532, -0.776, -0.338],
       [ 0.833, -0.408, -0.374],
       [ 0.153, -0.481,  0.864]]), translacion=array([-0.154, -0.093,  0.35 ]))
----------------------------------------------------------------------------------------------------
Esperado: [ 1.717  0.425  3.035  3.03   3.096 -2.396] y Configuracion(c1=True, c2=False, c3=True)
Obtenido: [ 2.026  0.27  -3.121 -2.848 -0.055  0.46 ] y Configuracion(c1=True, c2=False, c3=False)
Pose espera